# Datamine FXOUT Process Example

This notebook demonstrates how to configure and run the **`fxout`** process wrapper in `dmstudio`.

## Process Description

## FXOUT

Export a Datamine block model file to a Whittle FOUR-X format text file.

The following optional and required fields are expected in the block model file:

* TOLTON \- optional; minimum tonnage in parcel to be output on export (0.5)

* FORMAT \- optional; output format for economic file (0): 0 - fixed format, 1 - comma separated

* ELEMENT \- required; number of elements to be processed (1): limited to 10 for FOUR-X.

* ZONEFLD \- optional; prompt for optional ZONE field (0): 0 - don't prompt for ZONE field, 1 - prompt for ZONE field

#### Processing Methods

Select one of the following methods for specifying the tonnage and metal content of the cell or subcell:

* METHOD ASet the parcel name (type code) and the numeric tons and metal field names for each parcel type code specified. This is then repeated for each possible parcel type. This means that each parcel type's tons and metal content are contained in a separate fields. This is probably better for regularised block models, on record per block.

After specifying the method, you are prompted for the following information:

1. Maximum number of product parcels.

2. Default density

3. Merge identical product parcel types Y/N ?

4. Rock tonnes field name (ROCK)

5. Mining cost adjustment field name

6. Processing cost adjustment field name

7. Optionally set the 'Zone field name' if this option was selected in the Parameters tab.

8. Product parcel code (1-4 chars or blank to finish)

9. Tonnes field name (TONNES)

10. Metal content field (METAL)

11. Accept the data specification with "Y" and process.

* METHOD BDefine three fields (the parcel code field name, the parcel tons field name and the metal tonnes field name). This is much more in line with a subcelled block model which allows multiple records for each parent block. The best way of doing this is to calculate the tons and metal in each subcell and then accumulate on the parcel name field and IJK. This will give just one record for each parcel type for each parent-cell, reducing the size of the import model.

After specifying the method, you are prompted for the following information:

* Maximum number of product parcels.

* Default density

* Merge identical product parcel types Y/N ?

* Rock tonnes field name (ROCK)

* Mining cost adjustment field name

* Processing cost adjustment field name

* Optionally set the 'Zone field name' if this option was selected in the Parameters tab.

* Product parcel code (1-4 chars or blank to finish)

* Tonnes field name (TONNES)

* Metal content field (METAL)

* Ore type field name

* Accept the data specification with "Y" and process.

The exported FOUR-X model file has a record per block with the following information:

* Block coordinatesIX, IY, IZ

* Total tonnes of ROCK in the block Mining cost adjustment factor

* Processing cost adjustment factor

* An optional Zone number, which can be used to indicate the source of the block in the original model.

* Parcels. Zero or more parcels can be defined for each block on separate records. Up to 99 parcels are allowed.

Each parcel has the following information:

* Rock type code (1-4 characters). This should not contain a period('.'), be 'ROCK' or start with 'GR_'. It is case sensitive and should be left justified.

* Tonnage of the parcel Metal content of the parcel for up to 10 elements

* Metal content is measured in metal units, not grade.

A parcel is allowed to have a tonnage and zero metal content, to allow multiple 'waste' material types to be identified. The block dimensions are defined by the Datamine model cell structure. The rock type code must be supplied as an alphanumeric field. Only the first 4 characters are accepted. The required field names are supplied interactively.

An explicit field for tonnes of rock and metal content in the cell/sub-cell is required, together with optional explicit fields for mining cost adjustment, and processing cost adjustment. Where sub-cells are supplied, then block averages are calculated for the optional mining and processing cost adjustment factors.

The user can choose whether multiple sub-cells of the same rock type code within a cell are to be summed into one parcel or not. If sub-cells are being output as separate parcels, a limit can be set on the total number of parcels output for a block. If necessary, parcels are combined with the same rules as the FOUR-X FXRB reblocking program. If a sub-cell model is supplied it must be in sorted (IJK) order.

### Input Files:

* **in** (Undefined):
  Input model file.
  Required=Yes

### Output Files:

### Fields:

### Parameters:

* **tolton**:
  Minimum tonnage in parcel to be output (0.5)
  Range=Undefined
  Values=Undefined
  Default=0.5
  Required=No

* **format**:
  Output format for economic file (0) 0 - fixed format 1 - comma separated
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **element**:
  Number of elements to be processed (1) Limited to 10 for FOUR-X.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=Yes

* **zonefld**:
  Prompt for optional ZONE field (0) 0 - don't prompt for ZONE field 1 - prompt for ZONE
  field
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('fxout')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute fxout
print("Running fxout...")
dm_cmd.fxout(
    in_i='t_assays',  # required
    tolton_p=0.5,  # required
    format_p=0,  # required
    element_p='required_val',  # required
    # zonefld_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("fxout execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_fxout_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")